In [ ]:
import pandas as pd
data=pd.read_csv('/Users/macbook/Documents/My Projects/Data science/3/CourseData.csv')

In [ ]:
data.head()

In [ ]:
data.shape

In [ ]:
print('Row count of our data is{}'.format(data.shape[0]))

In [ ]:
data.columns

In [ ]:
data.iloc[data['no_of_previous_cancellations'].idxmax()]

In [ ]:
data.iloc[data['no_of_previous_cancellations'].idxmin()]

In [ ]:
data.iloc[data['avg_price_per_room'].idxmax()]

In [ ]:
data.iloc[data['avg_price_per_room'].idxmin()]

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.columns

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data['no_of_children'].value_counts().sort_values(ascending=False)

In [ ]:
sns.countplot(x='no_of_children',data=data)

In [ ]:
data['type_of_meal_plan'].value_counts().sort_values(ascending=False)

In [ ]:
mealtypelabels=data['type_of_meal_plan'].unique()

In [ ]:
plt.pie(data['type_of_meal_plan'].value_counts(),labels=mealtypelabels)
plt.show()

In [ ]:
data['room_type_reserved'].value_counts().sort_values(ascending=False)

In [ ]:
sns.countplot(x='room_type_reserved', data=data)
plt.show()

In [ ]:
data['required_car_parking_space'].value_counts().sort_values(ascending=False)

In [ ]:
sns.countplot(x='required_car_parking_space', data=data)
plt.show()

In [ ]:
data['booking_status'].value_counts().sort_values(ascending=False)

In [ ]:
bookinglabels = data['booking_status'].unique()
plt.pie(data['booking_status'].value_counts(),labels=bookinglabels)
plt.show()

In [ ]:
data.columns

In [ ]:
data = data[['no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights',
             'type_of_meal_plan', 'room_type_reserved', 'no_of_previous_cancellations',
             'avg_price_per_room', 'booking_status']]

In [ ]:
data.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()


In [ ]:
data['type_of_meal_plan'].head()

In [ ]:
data['type_of_meal_plan'] = labelencoder.fit_transform(data['type_of_meal_plan'])

In [ ]:
data['type_of_meal_plan'].head()

In [ ]:
data['room_type_reserved']= labelencoder.fit_transform(data['room_type_reserved'])
data['booking_status']= labelencoder.fit_transform(data['booking_status'])

In [ ]:
data.head()

In [ ]:
data.corr()['booking_status'].sort_values(ascending=True)

In [ ]:
sns.heatmap(data=data.corr(),annot=True)

In [ ]:
data.columns

In [ ]:
X = data[['no_of_adults', 'no_of_children', 'no_of_weekend_nights', 'no_of_week_nights',
          'type_of_meal_plan', 'room_type_reserved', 'no_of_previous_cancellations',
          'avg_price_per_room']]

y = data [['booking_status']]          

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test , y_train , y_test = train_test_split(X,y,test_size=0.3)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.fit_transform(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
def modelresults(predictions):
    print(f'Accuracy score of the model is {accuracy_score(y_test,predictions)}')

In [ ]:
from sklearn.linear_model import LogisticRegression
log_model= LogisticRegression()
log_model.fit(scaled_X_train,y_train)

In [ ]:
log_predictions = log_model.predict(scaled_X_test)
modelresults(log_predictions)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

knn = KNeighborsClassifier()
k_values = list(range(1, 30))
pipe = Pipeline(steps=[('knn', knn)])
param_grid = {'knn__n_neighbors': k_values}

cv_classifier = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
cv_classifier.fit(scaled_X_train, y_train)
print(cv_classifier.best_params_)

In [ ]:
pred_gridknn = cv_classifier.predict(scaled_X_test)
modelresults(pred_gridknn)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rfr_model = RandomForestClassifier()
n_estimators = [32, 65, 128]
bootstrap = [True, False]
oob_score = [True, False]
max_features = [2,3]  # Add this list as it was missing in the original code

param_grid_rfr = {
    'n_estimators': n_estimators,
    'max_features': max_features,
    'bootstrap': bootstrap,
    'oob_score': oob_score
}

grid_rfr = GridSearchCV(rfr_model, param_grid_rfr, cv=5, scoring='accuracy')
grid_rfr.fit(scaled_X_train, y_train)
print(grid_rfr.best_params_)


In [ ]:
predsrfc=grid_rfr.predict(scaled_X_test)
modelresults(predsrfc)

In [ ]:
import numpy as np
customerdata=np.array([2,1,1,2,1,1,0,110])

In [ ]:
def cancel_or_not(customer):
    prediction = cv_classifier.predict(customer.reshape(1, -1))
    if prediction == 1:
        return "won't cancel the booking"
    else:
        return "will cancel the booking"


In [ ]:
print('This customer{}according to the K Nearest Nighbors Model'.format(cancel_or_not(customerdata)))
